<a href="https://colab.research.google.com/github/NikitaBeker/titanic-survival-project/blob/main/notebooks/stage2_data_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install ydata-profiling kagglehub

import kagglehub
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ydata_profiling import ProfileReport

path = kagglehub.dataset_download("mobeenfatimah/titanic-dataset")
print(f"Путь к файлам датасета: {path}")

csv_files = glob.glob(os.path.join(path, "*.csv"))
if not csv_files:
    raise FileNotFoundError("CSV-файл не найден в папке с датасетом.")
file_path = csv_files[0]
print(f"Загружаем файл: {file_path}")

df = pd.read_csv(file_path)
print(f"Размер исходной таблицы: {df.shape}")

os.makedirs('reports', exist_ok=True)

# Общая статистика
print("\n--- Общая статистика ---")
print(df.describe(include='all').T)

# Проверка пропусков
print("\n--- Пропуски до обработки ---")
missing = df.isnull().sum()
print(missing[missing > 0])

# Заполнение пропусков (медианой для числовых, модой для категориальных)
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(include=['object', 'category']).columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print("\n--- Пропуски после обработки ---")
print(f"Осталось пропусков: {df.isnull().sum().sum()}")

# Проверка дубликатов и их удаление
duplicates_before = df.duplicated().sum()
print(f"\nДубликатов до удаления: {duplicates_before}")
df = df.drop_duplicates().reset_index(drop=True)
print(f"Дубликатов после удаления: {df.duplicated().sum()}")
print(f"Размер таблицы после удаления дубликатов: {df.shape}")

# Константные признаки
constant_cols = [col for col in df.columns if df[col].nunique() <= 1]
print(f"\nКонстантные признаки: {constant_cols if constant_cols else 'не обнаружены'}")

# Баланс классов
print("\n--- Баланс классов (survived) ---")
print(df['survived'].value_counts(normalize=True))
print(df['survived'].value_counts())

# Распределение числовых признаков
df[num_cols].hist(figsize=(15, 12), bins=30, edgecolor='black')
plt.suptitle('Распределение числовых признаков', fontsize=16)
plt.tight_layout()
plt.savefig('reports/numeric_distributions.png') # Сохраняем для GitHub
plt.close() # Закрываем график, чтобы не забивать память

# Boxplot для ключевых признаков
key_numeric = ['age', 'ticket_price', 'lifeboat_access', 'mobility_score']
for col in key_numeric:
    if col in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[col], color='skyblue')
        plt.title(f'Boxplot: {col}')
        plt.xlabel(col)
        plt.tight_layout()
        plt.savefig(f'reports/boxplot_{col}.png') # Сохраняем для GitHub
        plt.close()

        # Оценка выбросов по методу IQR
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        outliers = df[(df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR))]
        print(f"Признак '{col}': выбросов по IQR — {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)")

# Профиль исходной таблицы
print("\nГенерируем HTML-профиль таблицы (это может занять пару минут)...")
profile = ProfileReport(df, title="Titanic Dataset Profile", explorative=True)
profile.to_file("reports/titanic_profile_report.html")
print("Профиль сохранен в reports/titanic_profile_report.html")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.5/682.5 kB 8.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 19.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [ydata-profiling]


/tmp/ipykernel_3608/4234493339.py:10: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


Using Colab cache for faster access to the 'titanic-dataset' dataset.
Путь к файлам датасета: /kaggle/input/titanic-dataset
Загружаем файл: /kaggle/input/titanic-dataset/titanic_dataset.csv
Размер исходной таблицы: (100000, 31)

--- Общая статистика ---
                            count  unique           top   freq          mean  \
passenger_id               100000  100000     T2-099984      1           NaN   
full_name                  100000     576      Wei Chen    214           NaN   
age                      100000.0     NaN           NaN    NaN     34.791995   
gender                     100000       3          Male  49985           NaN   
nationality                100000      10           USA  24793           NaN   
marital_status             100000       4        Single  46318           NaN   
occupation                  91838       9       Student  25557           NaN   
class                    100000.0     NaN           NaN    NaN       2.45378   
ticket_number            1

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 31/31 [00:04<00:00,  6.55it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Профиль сохранен в reports/titanic_profile_report.html
